In [1]:
import pandas as pd
import numpy as np
import os, math
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold

from data_preprocess import process_dicom_series_safe
from dataset import RSNAAneurysmDataset, collate
from model import EffnetAneurysmClassifier
from metric import AverageMeter, auc_per_label, rsna_final_score
from utils import set_seed, LABEL_COLS

input_dir = "/home/khor/kaggle_kcw/kaggle-RSNA-Intracranial-Aneurysm-Detection/input/"
train_df = pd.read_csv(f"{input_dir}/train.csv") 
train_localizers_df = pd.read_csv(f"{input_dir}/train_localizers.csv")

INPUT_DIR = "/home/khor/kaggle_kcw/kaggle-RSNA-Intracranial-Aneurysm-Detection/input"
train_csv = os.path.join(INPUT_DIR, "train.csv")
train_df = pd.read_csv(train_csv)

TARGET_SHAPE = (32, 384, 384)

In [2]:
PREPROCESSED_DIR = os.path.join("/home/khor/kaggle_kcw/kaggle-RSNA-Intracranial-Aneurysm-Detection/experiment_1", "train_volumes")
os.makedirs(PREPROCESSED_DIR, exist_ok=True)

In [3]:
train_ds = RSNAAneurysmDataset(
    df=train_df,
    input_dir=PREPROCESSED_DIR, # Use the new directory with preprocessed volumes
    target_shape=TARGET_SHAPE,
    label_cols=LABEL_COLS
)


In [9]:
# --- Pre-process DICOMs to NumPy arrays ---

PREPROCESSED_DIR = os.path.join("/home/khor/kaggle_kcw/kaggle-RSNA-Intracranial-Aneurysm-Detection/experiment_1", "train_volumes")
os.makedirs(PREPROCESSED_DIR, exist_ok=True)

print("Starting DICOM pre-processing...")

# Get a list of all series UIDs to process
all_series_uids = train_df['SeriesInstanceUID'].unique()

# Check which files have already been processed
processed_uids = {f.split('.')[0] for f in os.listdir(PREPROCESSED_DIR)}
uids_to_process = [uid for uid in all_series_uids if uid not in processed_uids]

if not uids_to_process:
    print("All series have already been pre-processed.")
else:
    print(f"Processing {len(uids_to_process)} new series...")
    for series_uid in tqdm(uids_to_process, desc="Preprocessing DICOMs"):
        dicom_series_path = os.path.join(INPUT_DIR, "series", series_uid)
        
        # Process the DICOM series to a NumPy array
        volume = process_dicom_series_safe(dicom_series_path, TARGET_SHAPE)
        
        # Save the NumPy array
        np.save(os.path.join(PREPROCESSED_DIR, f"{series_uid}.npy"), volume)
        break


Starting DICOM pre-processing...
Processing 4348 new series...


Preprocessing DICOMs:   0%|          | 0/4348 [00:00<?, ?it/s]

In [10]:
volume

array([[[ 1,  3,  3, ...,  4,  4,  3],
        [ 3, 19, 21, ..., 23, 23, 21],
        [ 3, 19, 20, ..., 23, 23, 22],
        ...,
        [ 2, 15, 10, ..., 16, 15, 14],
        [ 3, 18, 17, ..., 15, 11, 12],
        [ 3, 20, 21, ..., 16, 13, 15]],

       [[ 1,  3,  3, ...,  4,  3,  3],
        [ 3, 20, 18, ..., 21, 19, 18],
        [ 3, 20, 20, ..., 20, 19, 16],
        ...,
        [ 2, 15, 13, ..., 13, 13, 11],
        [ 3, 17, 15, ..., 17, 15, 15],
        [ 3, 19, 18, ..., 18, 17, 16]],

       [[ 1,  3,  3, ...,  4,  3,  3],
        [ 3, 20, 19, ..., 20, 16, 18],
        [ 3, 21, 20, ..., 18, 17, 19],
        ...,
        [ 2, 13, 14, ..., 14, 14, 17],
        [ 2, 12, 12, ..., 15, 15, 18],
        [ 2, 13, 13, ..., 15, 15, 17]],

       ...,

       [[ 0,  1,  0, ...,  0,  1,  1],
        [ 1,  4,  1, ...,  1,  5,  5],
        [ 1,  3,  0, ...,  1,  6, 10],
        ...,
        [ 1,  4,  4, ...,  0,  0,  0],
        [ 0,  4,  3, ...,  0,  0,  0],
        [ 1,  4,  2, ...,  0,  0

In [3]:
train_ds = RSNAAneurysmDataset(
    df=train_df,
    input_dir=PREPROCESSED_DIR, # Use the new directory with preprocessed volumes
    target_shape=TARGET_SHAPE,
    label_cols=LABEL_COLS
)

NUM_LABELS = len(LABEL_COLS)
print(f"Number of labels: {NUM_LABELS}")

y_for_split = train_df[LABEL_COLS[0]].values
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
tr_idx, va_idx = next(iter(skf.split(train_df, y_for_split)))

train_subset = Subset(train_ds, tr_idx)
val_ds = RSNAAneurysmDataset(
    df=train_df.iloc[va_idx].reset_index(drop=True),
    input_dir=PREPROCESSED_DIR, # Use the new directory with preprocessed volumes
    target_shape=TARGET_SHAPE,
    label_cols=LABEL_COLS
)

train_loader = DataLoader(train_subset, batch_size=4, shuffle=True,
                            num_workers=4, pin_memory=True, collate_fn=collate, persistent_workers=True)
val_loader   = DataLoader(val_ds,     batch_size=4, shuffle=False,
                            num_workers=4, pin_memory=True, collate_fn=collate, persistent_workers=True)

Number of labels: 14


In [5]:
train_ds[33]

(tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],
 
         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],
 
         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],
 
         ...,
 
         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 